In [47]:
# Exercise 1 — Filtering

# Data:
import pandas as pd

df = pd.DataFrame({
    "name": ["A", "B", "C", "D"],
    "age": [25, 30, 22, 35],
    "salary": [50000, 60000, 45000, 70000]
})

"""
Task:

Return rows where:
    age > 25
    salary > 50000
"""
filtered = df[
    (df["age"] > 25) & 
    (df["salary"] > 50000)
    ]

filtered


,name,age,salary
1,B,30,60000
3,D,35,70000


In [19]:
# Exercise 2 — Column Creation

"""
Using the same df:

Create a new column:
    salary_k = salary / 1000
"""
df["salary_k"] = df["salary"]/1000

df


,name,age,salary,salary_k
0,A,25,50000,50.0
1,B,30,60000,60.0
2,C,22,45000,45.0
3,D,35,70000,70.0


In [45]:
#Exercise 3 — GroupBy

#Data:
df = pd.DataFrame({
    "user": ["A", "B", "A", "B", "C"],
    "amount": [100, 200, 50, 300, 400]
})

"""
Task:
    Compute total spend per user
"""

total_spend_user = df.groupby("user")["amount"].sum().reset_index()

total_spend_user


,user,amount
0,A,150
1,B,500
2,C,400


In [38]:
# Exercise 4 — Data Cleaning

# Data:
df = pd.DataFrame({
    "name": ["A", "B", "C", "D"],
    "age": [25, None, 30, None]
})

"""
Task:
    Remove rows with missing age
"""
df2 = df.copy()
df2 = df2.dropna(subset=["age"])


,name,age
0,A,25.0
2,C,30.0


In [40]:
# Exercise 5 — Merge

# Data:
df_users = pd.DataFrame({
    "user": ["A", "B", "C"],
    "country": ["US", "CA", "UK"]
})

df_orders = pd.DataFrame({
    "user": ["A", "A", "B", "C"],
    "amount": [100, 200, 300, 400]
})

"""
Task:
    Join the datasets so you get:
        user
        country
        amount
"""
df_merged = pd.merge(df_users, df_orders, on = "user", how="left")
df_merged
        

,user,country,amount
0,A,US,100
1,A,US,200
2,B,CA,300
3,C,UK,400


In [219]:
# Exercise 7 — Multi-Aggregation

# Data:
df = pd.DataFrame({
    "user": ["A", "B", "A", "B", "C", "A"],
    "amount": [100, 200, 50, 300, 400, 150]
})

"""
Task:
    For each user compute:
        total spend
        average spend
        number of transactions
"""

user_agg = df.groupby("user", as_index=False).agg(
    total_spend = ('amount',sum),
    average_spend = ('amount','mean'),
    number_of_transactions = ('amount', 'size'), 
    )

user_agg


,user,total_spend,average_spend,number_of_transactions
0,A,300,100.0,3
1,B,500,250.0,2
2,C,400,400.0,1


In [220]:
# Exercise 8 — GroupBy + Filter

"""
Using the same data:

    Return only users whose:
        total spend > 300
"""

total_spend = df.groupby("user")["amount"].sum().reset_index()

total_spend_filter = total_spend[total_spend["amount"] > 300]
total_spend_filter

# 1-step version
result = (
    df.groupby("user", as_index=False)["amount"]
      .sum()
      .rename(columns={"amount":"total_spend"})
      .query("total_spend > 300")
    )
result

# another 1-step Verson
result2 = (
    df.groupby("user", as_index=False)["amount"]
      .sum()
      .rename(columns={"amount": "total_spend"})
      .loc[lambda x: x["total_spend"] > 300 ]
    )

result2


,user,total_spend
1,B,500
2,C,400


In [226]:
# Exercise 9 — Ranking (Top Users)

"""
Task:
    Get top 2 users by total spend
"""
# By rank
top_2 =(
    df.groupby("user", as_index=False)["amount"]
      .sum()
      .rename(columns={"amount": "total_spend"})
      .assign(rank=lambda x: x["total_spend"].rank(ascending=False, method="dense"))
      .query("rank <= 2")
)
top_2


#By Sort.Values
top_2 = (
    df.groupby("user", as_index=False)["amount"]
      .sum()
      .rename(columns={"amount": "total_spend"})
      .sort_values(by="total_spend" , ascending=False)
      .head(2)
)
top_2



,user,total_spend
1,B,500
2,C,400


In [229]:
# Exercise 10 — Merge + Aggregate

#Data:
df_users = pd.DataFrame({
    "user": ["A", "B", "C", "D"],
    "country": ["US", "CA", "UK", "US"]
})

df_orders = pd.DataFrame({
    "user": ["A", "A", "B", "C", "C", "C"],
    "amount": [100, 200, 300, 400, 100, 50]
})

"""
Task:
    Compute:
        total spend per user
        include country

    Final output:
        user
        country
        total_spend
"""

df_merge_agg = (
    pd.merge(df_users, df_orders, on="user", how="left")
      .groupby(["user","country"], as_index=False)
      .agg(total_spend=("amount", "sum")
        )
)
df_merge_agg

,user,country,total_spend
0,A,US,300.0
1,B,CA,300.0
2,C,UK,550.0
3,D,US,0.0


In [230]:
# Exercise 11 — Conditional Feature Engineering

# Data:
df = pd.DataFrame({
    "user": ["A", "B", "C", "D"],
    "salary": [50000, 70000, 40000, 90000]
})

"""
    Task:
        Create column:
                "segment":
                    "low" ≤ 50k
                    "medium" > 50k and ≤ 80k
                    "high" > 80k
"""
# 3-step approach
df.loc[df["salary"] <= 50000, "segment"] = "low"
df.loc[(df["salary"] > 50000) & (df["salary"] <= 80000) , "segment"] = "medium"
df.loc[df["salary"] > 80000, "segment"] = "high"
df

# 1-step approach
df["segment"] = pd.cut(df["salary"], bins=[-float('inf'), 50000, 80000, float('inf')], labels=["low", "medium", "high"])
df

,user,salary,segment
0,A,50000,low
1,B,70000,medium
2,C,40000,low
3,D,90000,high


In [258]:
# Exercise 12 — Missing Data Strategy

# Data:

df = pd.DataFrame({
    "user": ["A", "A", "B", "B"],
    "age": [25, None, 30, None]
})

"""
Task:
    Fill missing ages using:
    average age per user
"""
# Multi-line solution
df_cleaned=df.copy()
df_cleaned['age'] = df_cleaned['age'].fillna(df_cleaned.groupby('user')['age'].transform('mean'))
df_cleaned

# One-line Soulution
df_cleaned = (
    df.assign(age= lambda x: x['age']
      .fillna(x.groupby('user')['age'].transform('mean')))
    )
df_cleaned

,user,age
0,A,25.0
1,A,25.0
2,B,30.0
3,B,30.0


In [266]:
# Exercise 13 — Date Handling

#Data:
df = pd.DataFrame({
    "date": ["2024-01-01", "2024-01-15", "2024-02-01"],
    "amount": [100, 200, 300]
})

"""
Task:
    Convert to datetime and:
                extract month
                compute total spend per month
"""

# multi-line Code
df["date"] = pd.to_datetime(df["date"])
df["year_month"] = df["date"].dt.to_period("M")
df_total_spend_month = df.groupby("year_month")["amount"].sum().reset_index()
df_total_spend_month

# One-line Code
df_total_spend_month= (
    df.assign(date=lambda x: pd.to_datetime(x["date"]))
      .assign(year_month=lambda x:x["date"].dt.to_period("M"))
      .groupby("year_month", as_index=False)["amount"]
      .sum()
    )
df_total_spend_month

,year_month,amount
0,2024-01,300
1,2024-02,300
